In [ ]:
import requests
import json
import pandas as pd
import time
from datetime import datetime, timedelta, timezone

In [ ]:
API_KEY = "MASKMASKMASKMASK"
city = "Seoul"

In [ ]:
url_city = f"http://api.openweathermap.org/geo/1.0/direct?q={city}&limit=5&appid={API_KEY}"

In [ ]:
response_city = requests.get(url_city);response_city

<Response [200]>

In [ ]:
response_city = json.loads(response_city.text)

In [ ]:
response_city

[{'name': 'Seoul',
  'local_names': {'lt': 'Seulas',
   'sr': 'Сеул',
   'my': 'ဆိုးလ်မြို့',
   'nl': 'Seoel',
   'mr': 'सोल',
   'et': 'Sŏul',
   'tr': 'Seul',
   'hy': 'Սեուլ',
   'os': 'Сеул',
   'ba': 'Сеул',
   'es': 'Seúl',
   'hr': 'Seul',
   'ko': '서울',
   'fi': 'Soul',
   'ar': 'سول',
   'am': 'ሶል',
   'ro': 'Seul',
   'uz': 'Seul',
   'bg': 'Сеул',
   'cv': 'Сеул',
   'it': 'Seul',
   'tk': 'Seul',
   'hu': 'Szöul',
   'fa': 'سئول',
   'fr': 'Séoul',
   'la': 'Seulum',
   'ml': 'സോൾ',
   'ka': 'სეული',
   'bh': 'सियोल',
   'ur': 'سؤل',
   'bn': 'সিওল',
   'pt': 'Seul',
   'sk': 'Soul',
   'bs': 'Seul',
   'an': 'Seúl',
   'sl': 'Seul',
   'eo': 'Seulo',
   'az': 'Seul',
   'hi': 'सियोल',
   'lv': 'Seula',
   'kk': 'Сеул',
   'uk': 'Сеул',
   'ja': 'ソウル',
   'en': 'Seoul',
   'vi': 'Seoul',
   'sv': 'Seoul',
   'be': 'Сеул',
   'qu': 'Siul',
   'kn': 'ಸೌಲ್',
   'ca': 'Seül',
   'ky': 'Сеул',
   'mn': 'Сөүл',
   'zh': '首尔市 / 首爾',
   'km': 'សេអ៊ូល',
   'is': 'Seúl',
   'ku': 'S

In [ ]:
test_lat = response_city[0]['lat']

In [ ]:
test_lon = response_city[0]['lon']

In [ ]:
response_coord = requests.get(hist_url);response_coord

<Response [401]>

In [ ]:
url = f"https://api.openweathermap.org/data/2.5/weather?q=Seoul&appid={API_KEY}&units=metric"
response = requests.get(url)
print(f"현재 날씨 API - Status: {response.status_code}")
print(f"Response: {response.text}")


현재 날씨 API - Status: 200
Response: {"coord":{"lon":126.9778,"lat":37.5683},"weather":[{"id":803,"main":"Clouds","description":"broken clouds","icon":"04n"}],"base":"stations","main":{"temp":0.76,"feels_like":0.76,"temp_min":0.76,"temp_max":1.78,"pressure":1028,"humidity":64,"sea_level":1028,"grnd_level":1017},"visibility":10000,"wind":{"speed":1.03,"deg":280},"clouds":{"all":75},"dt":1770111265,"sys":{"type":1,"id":8105,"country":"KR","sunrise":1770071671,"sunset":1770109027},"timezone":32400,"id":1835848,"name":"Seoul","cod":200}


In [ ]:
url_coord = f"https://api.openweathermap.org/data/2.5/weather?lat={test_lat}&lon={test_lon}&appid={API_KEY}&units=metric&lang=kr"

In [ ]:
response_coord = requests.get(url_coord);response_coord

<Response [200]>

In [ ]:
response_coord = json.loads(response_coord.text);response_coord

{'coord': {'lon': 126.9783, 'lat': 37.5667},
 'weather': [{'id': 800, 'main': 'Clear', 'description': '맑음', 'icon': '01d'}],
 'base': 'stations',
 'main': {'temp': 3.76,
  'feels_like': 0.95,
  'temp_min': 3.76,
  'temp_max': 3.78,
  'pressure': 1028,
  'humidity': 48,
  'sea_level': 1028,
  'grnd_level': 1018},
 'visibility': 10000,
 'wind': {'speed': 3.09, 'deg': 300},
 'clouds': {'all': 0},
 'dt': 1770101698,
 'sys': {'type': 1,
  'id': 8105,
  'country': 'KR',
  'sunrise': 1770071671,
  'sunset': 1770109027},
 'timezone': 32400,
 'id': 1835848,
 'name': 'Seoul',
 'cod': 200}

In [ ]:
import requests
from langchain_core.tools import tool
from datetime import datetime, timedelta

# --- 부산 기본 좌표 (챗봇 기본 지역) ---
DEFAULT_CITY = {"name": "부산", "lat": 35.1796, "lon": 129.0756}

# --- 헬퍼 함수 ---

def get_coordinates(city: str) -> dict:
    url = f"http://api.openweathermap.org/geo/1.0/direct?q={city}&limit=1&appid={API_KEY}"

    try:
        response = requests.get(url, timeout=5)
        if response.status_code == 200 and response.json():
            data = response.json()[0]
            name = data.get("local_names", {}).get("ko", data["name"])
            return {"name": name, "lat": data["lat"], "lon": data["lon"]}
        return {"error": f"'{city}' 도시를 찾을 수 없음.", **DEFAULT_CITY}
    except Exception as e:
        return {"error": f"geo API 오류: {e}.", **DEFAULT_CITY}

def get_weather(city: str) -> dict:
    """현재 날씨 수집"""
    coord = get_coordinates(city)

    url = f"https://api.openweathermap.org/data/2.5/weather?lat={coord['lat']}&lon={coord['lon']}&appid={API_KEY}&units=metric&lang=kr"
    response = requests.get(url, timeout=5)

    if response.status_code != 200:
        return {"error": f"Weather API 에러: {response.status_code}"}

    data = response.json()
    return {
        "coord": coord,
        "temp": data["main"]["temp"],
        "feels_like": data["main"]["feels_like"],
        "humidity": data["main"]["humidity"],
        "description": data["weather"][0]["description"],
        "wind_speed": data["wind"]["speed"],
        "rain": data.get("rain", {}).get("1h", 0),
        "snow": data.get("snow", {}).get("1h", 0),
        "clouds": data["clouds"]["all"],
    }


def get_forecast(city: str) -> dict:
    """5일간 3시간 간격 예보 수집"""
    coord = get_coordinates(city)

    url = f"https://api.openweathermap.org/data/2.5/forecast?lat={coord['lat']}&lon={coord['lon']}&appid={API_KEY}&units=metric&lang=kr"
    response = requests.get(url, timeout=5)

    if response.status_code != 200:
        return {"error": f"Forecast API 에러: {response.status_code}"}

    data = response.json()
    forecasts = []
    for item in data["list"]:
        forecasts.append({
            "일시": item["dt_txt"],
            "섭씨": item["main"]["temp"],
            "체감": item["main"]["feels_like"],
            "습도": item["main"]["humidity"],
            "날씨": item["weather"][0]["description"],
            "풍속": item["wind"]["speed"],
            "강수량": item.get("rain", {}).get("3h", 0),
            "적설량": item.get("snow", {}).get("3h", 0) / 10,
            "구름양": item["clouds"]["all"],
        })

    return {"coord": coord, "forecasts": forecasts}


def get_history(city: str, days_ago: int = 1, hours: int = 24) -> dict | None:
    """과거 날씨 데이터 조회 (학생 플랜: 최대 1주일)"""
    from datetime import datetime, timedelta, timezone

    KST = timezone(timedelta(hours=9))

    # 좌표 먼저 조회
    coord = get_coordinates(city)
    if not coord:
        return None

    # start 타임스탬프 계산: 해당 날짜 00:00 KST
    now_kst = datetime.now(KST)
    target_start = (now_kst.replace(hour=0, minute=0, second=0, microsecond=0)
                    - timedelta(days=days_ago))
    start_unix = int(target_start.timestamp())

    url = (
        f"https://history.openweathermap.org/data/2.5/history/city"
        f"?lat={coord['lat']}&lon={coord['lon']}"
        f"&type=hour"
        f"&start={start_unix}"
        f"&cnt={hours}"
        f"&appid={API_KEY}"
        f"&units=metric"
        f"&lang=kr"
    )

    try:
        response = requests.get(url)
        if response.status_code == 200:
            return response.json()
        else:
            print(f"History API error: {response.status_code} - {response.text}")
            return None
    except requests.RequestException as e:
        print(f"Request failed: {e}")
        return None


def parse_history(data: dict) -> dict:
    """History API 응답을 정리된 형태로 변환"""
    if not data or "list" not in data:
        return {}

    hourly = []
    for item in data["list"]:
        from datetime import datetime, timezone, timedelta
        KST = timezone(timedelta(hours=9))

        dt_kst = datetime.fromtimestamp(item["dt"], tz=KST)

        hourly.append({
            "time": dt_kst.strftime("%H:%M"),
            "temp": item["main"]["temp"],
            "humidity": item["main"]["humidity"],
            "description": item["weather"][0]["description"],
            "wind_speed": item["wind"]["speed"],
            "rain_1h": item.get("rain", {}).get("1h", 0.0),
            "snow_1h": item.get("snow", {}).get("1h", 0.0),
            "clouds": item["clouds"]["all"],
        })

    # 일별 요약 (days_ago가 여러 날일 때)
    temps = [h["temp"] for h in hourly]

    summary = {
        "max_temp": max(temps) if temps else None,
        "min_temp": min(temps) if temps else None,
        "total_rain_mm": round(sum(h["rain_1h"] for h in hourly), 1),
        "total_snow_mm": round(sum(h["snow_1h"] for h in hourly), 1),
        "hourly": hourly,
    }

    return summary

@tool
def forecast_tool(city: str, days: int = 1) -> str:
    """
    도시명과 일수를 받아 예보 날씨 정보를 반환한다.

    Args:
        city: 날씨를 조회할 도시명 (예: "부산", "서울", "도쿄")
        days: 미래의 날씨 (예: 1은 내일, 2는 모레, 3은 글모레)
    """
    data = get_forecast(city)
    if "error" in data:
        return data["error"]

    coord = data["coord"]
    forecasts = data["forecasts"]

    target_date = (datetime.now() + timedelta(days=days)).strftime("%Y-%m-%d")
    filtered = [f for f in forecasts if f["일시"].startswith(target_date)]

    if not filtered:
        return f"'{target_date}' 예보 데이터가 없습니다."

    # 하루치 기준으로 최고·최저·강수·적설 계산
    max_temp = max(f["섭씨"] for f in filtered)
    min_temp = min(f["섭씨"] for f in filtered)
    total_rain = sum(f["강수량"] for f in filtered)
    total_snow = sum(f["적설량"] for f in filtered)

    prefix = f"⚠️ {coord['error']}\n" if "error" in coord else ""

    # 요약
    result = (
        f"{prefix}{coord['name']} {target_date} 날씨 예보:\n\n"
        f"📊 요약\n"
        f"- 최고기온: {max_temp}°C\n"
        f"- 최저기온: {min_temp}°C\n"
        f"- 총 강수량: {total_rain}mm\n"
        f"- 총 적설량: {total_snow}cm\n\n"
    )

    # 시간별 상세
    for f in filtered:
        result += (
            f"[{f['일시']}]\n"
            f"- 온도: {f['섭씨']}°C (체감 {f['체감']}°C)\n"
            f"- 습도: {f['습도']}%\n"
            f"- 날씨: {f['날씨']}\n"
            f"- 풍속: {f['풍속']} m/s\n"
            f"- 강수량: {f['강수량']}mm\n"
            f"- 적설량: {f['적설량']}cm\n"
            f"- 구름양: {f['구름양']}%\n\n"
        )
    return result

# --- LangGraph Tool ---

@tool
def weather_tool(city: str) -> str:
    """
    도시명을 받아 현재 날씨 정보를 반환한다.

    Args:
        city: 날씨를 조회할 도시명 (예: "부산", "서울", "도쿄")
    """
    weather = get_weather(city)  # city로 바로 호출

    if "error" in weather:
        return weather["error"]

    coord = weather.pop("coord")  # coord 분리
    prefix = f"⚠️ {coord['error']}\n" if "error" in coord else ""

    return (
        f"{prefix}"
        f"{coord['name']} 현재 날씨:\n"
        f"- 온도: {weather['temp']}°C (체감 {weather['feels_like']}°C)\n"
        f"- 습도: {weather['humidity']}%\n"
        f"- 날씨: {weather['description']}\n"
        f"- 풍속: {weather['wind_speed']} m/s\n"
        f"- 강수량: {weather['rain']}mm"
        f"- 적설량: {weather['snow']}mm"
        f"- 구름양: {weather['clouds']}%"
    )

@tool
def history_tool(city: str, days_ago: int = 1) -> str:
    """
    과거 날씨 데이터를 조회하는 툴.

    Args:
        city: 도시명 (한글 가능, 예: "서울", "부산")
        days_ago: 몇 날 전의 날씨 (1=어제, 2=그제, ... 최대 7)
    """
    if days_ago < 1 or days_ago > 7:
        return "학생 플랜은 최대 1주일(7일) 이내의 과거 날씨만 조회 가능합니다."

    data = get_history(city, days_ago=days_ago)
    if not data:
        return f"{city}의 과거 날씨 데이터를 가져올 수 없습니다."

    parsed = parse_history(data)
    if not parsed:
        return "데이터 파싱에 실패했습니다."

    # KST 날짜 계산
    from datetime import datetime, timedelta, timezone
    KST = timezone(timedelta(hours=9))
    target_date = (datetime.now(KST) - timedelta(days=days_ago)).strftime("%Y-%m-%d")

    result = (
        f"📅 {city} - {target_date} 날씨\n"
        f"🌡️ 최고기온: {parsed['max_temp']}°C / 최저기온: {parsed['min_temp']}°C\n"
        f"🌧️ 총강수량: {parsed['total_rain_mm']}mm\n"
        f"❄️ 총적설량: {parsed['total_snow_mm']}mm\n\n"
        f"시간별 상세:\n"
    )

    for h in parsed["hourly"]:
        result += f"  {h['time']} | {h['temp']}°C | {h['description']} | 풍속 {h['wind_speed']}m/s\n"

    return result

In [ ]:
result = weather_tool.invoke({"city": "울산"})
print(result)

울산 현재 날씨:
- 온도: 1.06°C (체감 -1.01°C)
- 습도: 44%
- 날씨: 튼구름
- 풍속: 1.83 m/s
- 강수량: 0mm- 적설량: 0mm- 구름양: 54%


In [ ]:
tomorrow = forecast_tool.invoke({"city": "인천","days":1})
print(tomorrow)

인천 2026-02-04 날씨 예보:

📊 요약
- 최고기온: 4.03°C
- 최저기온: -0.04°C
- 총 강수량: 0mm
- 총 적설량: 0.0cm

[2026-02-04 00:00:00]
- 온도: -0.04°C (체감 -3.65°C)
- 습도: 74%
- 날씨: 온흐림
- 풍속: 3.12 m/s
- 강수량: 0mm
- 적설량: 0.0cm
- 구름양: 93%

[2026-02-04 03:00:00]
- 온도: 1.65°C (체감 -1.64°C)
- 습도: 69%
- 날씨: 온흐림
- 풍속: 3.15 m/s
- 강수량: 0mm
- 적설량: 0.0cm
- 구름양: 100%

[2026-02-04 06:00:00]
- 온도: 4.03°C (체감 1.46°C)
- 습도: 69%
- 날씨: 온흐림
- 풍속: 2.85 m/s
- 강수량: 0mm
- 적설량: 0.0cm
- 구름양: 100%

[2026-02-04 09:00:00]
- 온도: 2.39°C (체감 1.2°C)
- 습도: 80%
- 날씨: 온흐림
- 풍속: 1.35 m/s
- 강수량: 0mm
- 적설량: 0.0cm
- 구름양: 85%

[2026-02-04 12:00:00]
- 온도: 1.85°C (체감 0.35°C)
- 습도: 85%
- 날씨: 튼구름
- 풍속: 1.5 m/s
- 강수량: 0mm
- 적설량: 0.0cm
- 구름양: 63%

[2026-02-04 15:00:00]
- 온도: 1.41°C (체감 -0.81°C)
- 습도: 87%
- 날씨: 온흐림
- 풍속: 2 m/s
- 강수량: 0mm
- 적설량: 0.0cm
- 구름양: 98%

[2026-02-04 18:00:00]
- 온도: 1.08°C (체감 -1.43°C)
- 습도: 87%
- 날씨: 온흐림
- 풍속: 2.2 m/s
- 강수량: 0mm
- 적설량: 0.0cm
- 구름양: 95%

[2026-02-04 21:00:00]
- 온도: 0.97°C (체감 -1.56°C)
- 습도: 86%
- 날씨: 온흐림
- 풍속: 2.2 m/s
- 강수

In [ ]:
print(history_tool.invoke({"city": "거제", "days_ago": 3}))

📅 거제 - 2026-01-31 날씨
🌡️ 최고기온: 5.25°C / 최저기온: -0.08°C
🌧️ 총강수량: 0.0mm
❄️ 총적설량: 0.0mm

시간별 상세:
  00:00 | 0.61°C | overcast clouds | 바람 3.49m/s
  01:00 | 0.26°C | overcast clouds | 바람 3.49m/s
  02:00 | -0.02°C | overcast clouds | 바람 3m/s
  03:00 | -0.02°C | overcast clouds | 바람 2.93m/s
  04:00 | 0.05°C | overcast clouds | 바람 3.36m/s
  05:00 | -0.08°C | overcast clouds | 바람 3.7m/s
  06:00 | -0.07°C | overcast clouds | 바람 3.77m/s
  07:00 | -0.01°C | overcast clouds | 바람 3.66m/s
  08:00 | -0.03°C | overcast clouds | 바람 3.26m/s
  09:00 | 0.48°C | overcast clouds | 바람 3.01m/s
  10:00 | 1.26°C | overcast clouds | 바람 2.84m/s
  11:00 | 2.25°C | overcast clouds | 바람 2.41m/s
  12:00 | 3.46°C | overcast clouds | 바람 2.1m/s
  13:00 | 4.35°C | broken clouds | 바람 1.93m/s
  14:00 | 4.96°C | broken clouds | 바람 2.2m/s
  15:00 | 5.25°C | broken clouds | 바람 2.32m/s
  16:00 | 5.13°C | clear sky | 바람 2.45m/s
  17:00 | 5.04°C | clear sky | 바람 1.94m/s
  18:00 | 4.75°C | few clouds | 바람 1.69m/s
  19:00 | 4.58°C | 